In [7]:
# imports
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import imaplib, email
from bs4 import BeautifulSoup
import re
from time import sleep
import pandas as pd
import math
import yaml

# options
with open('/Users/pickle/Documents/code_is_life/credit-card/config.yml', 'r') as stream:
    try:
        config = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)
options = webdriver.ChromeOptions()
options.add_argument("user-data-dir={}".format(config['user-data-dir']))

# driver
driver = webdriver.Chrome(ChromeDriverManager().install(), options=options); sleep(5)



====== WebDriver manager ======
Current google-chrome version is 92.0.4515
Get LATEST driver version for 92.0.4515
Get LATEST driver version for 92.0.4515
Trying to download new driver from https://chromedriver.storage.googleapis.com/92.0.4515.107/chromedriver_mac64.zip
Driver has been saved in cache [/Users/pickle/.wdm/drivers/chromedriver/mac64/92.0.4515.107]


In [8]:
# function to search for a key value pair 
def search(client, key, value): 
    result, _bytes = client.search(None, key, '"{}"'.format(value))
    return _bytes
# login to CC
driver.get(config['url']['login']); sleep(5) 
driver.find_element_by_id('username').send_keys(config['cc']['member-number']); sleep(5) 
driver.find_element_by_id('password').send_keys(config['cc']['password']); sleep(5) 
driver.find_element_by_id('loginButton').send_keys(Keys.RETURN); sleep(10)
if driver.current_url == config['url']['mfa']:
    driver.find_element_by_name('sendOTP').send_keys(Keys.RETURN); sleep(60)
    # login to Gmail
    client = imaplib.IMAP4_SSL('imap.gmail.com') 
    client.login(config['email']['address'], config['email']['password']) 
    _ = client.select('Inbox') 
    uids = search(client, 'FROM', config['cc']['mfa-email'])[0].split()
    _, latest_email_bytes = client.fetch(uids[-1], '(RFC822)')
    latest_email_text = str(latest_email_bytes[0][1])
    soup = BeautifulSoup(latest_email_text, 'lxml')
    pattern_1 = re.compile(r'passcode is (\d+)')
    code = pattern_1.findall(soup.find_all(text=pattern_1)[0])[0]
    # 2-factor
    driver.find_element_by_id('mfaCodeInputField').send_keys(code); sleep(5)
    driver.find_element_by_name('registerDevice').send_keys(Keys.RETURN); sleep(5)
    driver.refresh(); sleep(5)

WebDriverException: Message: chrome not reachable
  (Session info: chrome=92.0.4515.159)


In [ ]:
import urllib.request, json 
from tqdm import tqdm
import numpy as np
#!pip install editdistance
import editdistance

In [ ]:
surahInfo = json.load(open('surahInfo.json', 'r'))

In [ ]:
prefixes = ['aal', 'ad', 'adh', 'al', 'an', 'ar', 'as', 'ash', 'at', 'az']
def text_cleaned(text):
    _text = text
    for prefix in prefixes:
        if _text.startswith(prefix):
            if '-' in _text:
                return _text.replace(prefix+'-', '')
            #else:
            #    return _text.replace(prefix, '')
    else:
        return _text
names_cleaned = [text_cleaned(item["englishName"].lower()) for item in surahInfo['data']]

In [ ]:
def func(guess):
    guess_lower = guess.lower()
    lst = []
    for i, item in enumerate(surahInfo['data']):
        name = item['englishName']
        name_lower = name.lower()
        name_cleaned = names_cleaned[i]
        if guess_lower in name_lower:
            lst.append((name, 0, name_lower))
        else:
            guess_cleaned = text_cleaned(guess_lower)
            #lst.append((name, editdistance.eval(name_lower, guess_lower)))
            lst.append((name, editdistance.eval(name_cleaned, guess_cleaned), name_cleaned))
    return sorted(lst, key = lambda x: (x[1], x[2]))

In [ ]:
func("alaa")

In [ ]:
for item in tqdm(foo['chapters']):
    id_ = item["id"]
    verses_count = item["verses_count"]
    for verse_count in range(1, verses_count + 1):
        with urllib.request.urlopen("https://api.quranwbw.com/{}/{}".format(id_, verse_count)) as url:
            data = json.loads(url.read().decode())
            with open('Test/{}_{}.json'.format(id_, verse_count), 'w') as fp:
                json.dump(data, fp)

100%|██████████| 114/114 [15:36<00:00,  8.21s/it]


In [ ]:
data.keys()

dict_keys(['ayahs', 'meta', 'words'])

In [ ]:
import os
yep = {'word_arabic_uthmani', 'word_translation', 'word_transliteration'}
for i in tqdm(range(1, 114+1)): 
    for v in range(1, foo["chapters"][i-1]['verses_count'] + 1):
        data = json.load(open("Test/{}_{}.json".format(i, v), 'r'))
        data['ayahs'] = {'ayah_uthmani': data['ayahs']['ayah_uthmani']}
        del data['meta']
        data['words'] = [{key: value for key, value in word_dict.items() if key in yep} for word_dict in data['words']]
        json.dump(data, open('Test_cleaned/{}_{}.json'.format(i, v), 'w'))

100%|██████████| 114/114 [00:17<00:00,  6.69it/s]


In [ ]:
data

{'ayahs': {'ayah_uthmani': 'مِنَ ٱلۡجِنَّةِ وَٱلنَّاسِ'},
 'words': [{'word_arabic_uthmani': 'مِنَ',
   'word_translation': 'From',
   'word_transliteration': 'mina'},
  {'word_arabic_uthmani': 'ٱلۡجِنَّةِ',
   'word_translation': 'the jinn',
   'word_transliteration': 'al-jinati'},
  {'word_arabic_uthmani': 'وَٱلنَّاسِ',
   'word_translation': 'and men',
   'word_transliteration': 'wal-nasi'}]}

In [ ]:
data = json.load(open("Test/{}_{}.json".format(1, 1), 'r'))

In [ ]:
json.dump(data, open('Test_cleaned/1_1.json', 'w'))

In [ ]:
data['words'][0]

{'word_arabic': 'بِسْمِ',
 'word_arabic_indopak': 'بِسْمِ',
 'word_arabic_uthmani': 'بِسۡمِ',
 'word_ayah': 1,
 'word_corpus': {'forms': {'active_participle': 'مُسَمٍّ',
   'imperative': 'سَمِّ',
   'imperfect': 'يُسَمِّي',
   'lemma': 'اسْم',
   'passive_participle': 'مُسَمًّى',
   'perfect': 'سَمّٰى',
   'verbal_noun': 'تَسْمِيَةً'},
  'grammar': [{'text': 'بِ', 'type': 'preposition'},
   {'text': 'سْمِ', 'type': 'noun'}]},
 'word_number_in_ayah': 1,
 'word_number_in_quran': 1,
 'word_number_in_surah': 1,
 'word_root': 'س م و',
 'word_surah': 1,
 'word_translation': 'In (the) name',
 'word_transliteration': "bis'mi"}

In [ ]:
import http.client
from tqdm import tqdm

In [ ]:
conn = http.client.HTTPSConnection("api.quran.com")
payload = "{}"
url = "/api/v4/resources/translations/{}/info"
found = set()
for i in tqdm(range(0, 200)):
    conn.request("GET", url.format(i), payload)
    res = conn.getresponse()
    data = json.loads(res.read().strip())
    if 'error' not in data:
        print(i)
        found.add(i)

 10%|█         | 21/200 [00:05<00:52,  3.41it/s]19
20
 12%|█▏        | 24/200 [00:06<00:48,  3.62it/s]22
23
 14%|█▎        | 27/200 [00:07<00:41,  4.19it/s]25
26
 14%|█▍        | 29/200 [00:07<00:29,  5.74it/s]27
28
 16%|█▌        | 31/200 [00:07<00:24,  6.98it/s]29
30
 16%|█▋        | 33/200 [00:07<00:21,  7.83it/s]31
32
 18%|█▊        | 37/200 [00:08<00:31,  5.20it/s]35
36
 20%|█▉        | 39/200 [00:08<00:24,  6.59it/s]37
38
 22%|██▏       | 43/200 [00:09<00:29,  5.26it/s]41
42
 22%|██▎       | 45/200 [00:10<00:23,  6.64it/s]43
44
 24%|██▎       | 47/200 [00:10<00:20,  7.55it/s]45
46
 24%|██▍       | 49/200 [00:10<00:18,  8.16it/s]47
48
 26%|██▌       | 51/200 [00:10<00:18,  8.19it/s]49
50
 26%|██▋       | 53/200 [00:10<00:17,  8.33it/s]51
52
 28%|██▊       | 55/200 [00:11<00:16,  8.58it/s]53
54
 28%|██▊       | 57/200 [00:11<00:16,  8.69it/s]55
56
 38%|███▊      | 76/200 [00:17<00:23,  5.26it/s]74
75
 39%|███▉      | 78/200 [00:17<00:18,  6.54it/s]76
77
 40%|████      | 80/200 [00:

In [ ]:
conn = http.client.HTTPSConnection("api.quran.com")
payload = "{}"
url = "/api/v4/resources/translations/{}/info"
for i in found:
    conn.request("GET", url.format(i), payload)
    res = conn.getresponse()
    data = json.loads(res.read().strip())
    print(data)

{'info': {'id': 17, 'info': None}}
{'info': {'id': 19, 'info': None}}
{'info': {'id': 20, 'info': None}}
{'info': {'id': 22, 'info': None}}
{'info': {'id': 23, 'info': None}}
{'info': {'id': 25, 'info': None}}
{'info': {'id': 26, 'info': None}}
{'info': {'id': 27, 'info': None}}
{'info': {'id': 28, 'info': None}}
{'info': {'id': 29, 'info': None}}
{'info': {'id': 30, 'info': None}}
{'info': {'id': 31, 'info': ''}}
{'info': {'id': 32, 'info': None}}
{'info': {'id': 33, 'info': None}}
{'info': {'id': 35, 'info': None}}
{'info': {'id': 36, 'info': None}}
{'info': {'id': 37, 'info': None}}
{'info': {'id': 38, 'info': None}}
{'info': {'id': 39, 'info': None}}
{'info': {'id': 41, 'info': None}}
{'info': {'id': 42, 'info': ''}}
{'info': {'id': 43, 'info': None}}
{'info': {'id': 44, 'info': ''}}
{'info': {'id': 45, 'info': None}}
{'info': {'id': 46, 'info': None}}
{'info': {'id': 47, 'info': None}}
{'info': {'id': 48, 'info': None}}
{'info': {'id': 49, 'info': None}}
{'info': {'id': 50, 'info'

In [8]:
class Solution:
    count = 0
    def findTargetSumWays(self, nums, target):
        self.helper(nums, 0, 0, target)
        return self.count

    def helper(self, nums, i, t, target):
        if i == len(nums):
            if t == target:
                self.count += 1
        else:
            self.helper(nums, i+1, t+nums[i], target)
            self.helper(nums, i+1, t-nums[i], target)

In [9]:
s = Solution()
s.findTargetSumWays([1,1,1,1,1], 3)

5

In [18]:
class Solution:
    def maxArea(self, height):
        n = len(height)
        i, j = 0, n-1
        area = -1
        while True:
            area = max(area, (j-i)*min(height[i], height[j]))
        count = 0
        for i in range(n-1):
            if best_i == None or height[i] > height[best_i]:
                for j in range(i+1, n):
                    count += 1
                    area = (j-i)*min(height[i], height[j])
                    if area > best_area:
                        best_area = area
                        best_i = i
                        best_j = j
        print(count)
        '''
        return best_area


In [19]:
s = Solution()
s.maxArea([1,2,1])

2